# Goodgorithm — Political-content classifier training

Trains a TF-IDF + logistic regression classifier that scores a post's likelihood of being political content — a **binary** target (`non-political` vs. `political`), not a 5th feed category. See CLAUDE.md's Category filtering section for why political content stays out of `taxonomy.py`'s 4-category set.

**Training data**: `cardiffnlp/tweet_topic_multi`'s `news_&_social_concern` label as a broad, noisy pretraining base, combined with 60% of the hand-labelled political/civic-tone evaluation set (`training/political_tone_eval.jsonl`, whose `civic-neutral`/`civic-warm`/`civic-negative` labels collapse into `political`). The remaining 40% is held out and evaluated against below.

**No GPU needed** — TF-IDF + logistic regression trains on CPU in well under a minute.

**Before running:** this notebook fetches `processing/src/util/text_normalize.py` from a pinned commit, not a live fetch from `main`, so normalization can't drift between training and inference even though they run in different environments. If that file changes, update `TEXT_NORMALIZE_COMMIT` below.

In [ ]:
!pip install -q datasets scikit-learn skl2onnx onnxruntime boto3

## Fetch the shared text-normalization file

In [ ]:
TEXT_NORMALIZE_COMMIT = "525b1888e0158a0fb45cd385eff9799531abbdec"

import urllib.request

url = (
    f"https://raw.githubusercontent.com/goodgorithm/goodgorithm/"
    f"{TEXT_NORMALIZE_COMMIT}/processing/src/util/text_normalize.py"
)
urllib.request.urlretrieve(url, "text_normalize.py")

import text_normalize

print("fetched text_normalize.py @", TEXT_NORMALIZE_COMMIT)

## Load the political/civic-tone evaluation set

In [ ]:
import json
import random
from collections import Counter, defaultdict

random.seed(42)

DATA_PATH = "political_tone_eval.jsonl"  # copy training/political_tone_eval.jsonl next to this notebook before running

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]
for r in rows:
    r["is_political"] = r["label"] != "non-political"

by_label = defaultdict(list)
for r in rows:
    by_label[r["label"]].append(r)

train_233, test_233 = [], []
for label, items in by_label.items():
    items = items[:]
    random.shuffle(items)
    cut = int(len(items) * 0.6)
    train_233 += items[:cut]
    test_233 += items[cut:]
random.shuffle(train_233)
random.shuffle(test_233)
print(f"#233 split: train={len(train_233)} test(check)={len(test_233)}")
print("check-set label distribution:", Counter(r["label"] for r in test_233))

## Load `cardiffnlp/tweet_topic_multi`

Broader pretraining pool. `political` = 1 if the `news_&_social_concern` label is present, else 0 — the closest available proxy, though it covers general news/social issues rather than strictly political content, so it's noisier than the hand-labelled evaluation set. Only `train_2020`/`validation_2020` are used; `test_2021` isn't needed since the evaluation set above is this task's real ground truth.

In [ ]:
from datasets import load_dataset

ds = load_dataset("cardiffnlp/tweet_topic_multi")

ORIGINAL_LABELS = [
    "arts_&_culture", "business_&_entrepreneurs", "celebrity_&_pop_culture",
    "diaries_&_daily_life", "family", "fashion_&_style", "film_tv_&_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_&_educational",
    "music", "news_&_social_concern", "other_hobbies", "relationships",
    "science_&_technology", "sports", "travel_&_adventure", "youth_&_student_life",
]
NEWS_IDX = ORIGINAL_LABELS.index("news_&_social_concern")

cardiff_texts, cardiff_labels = [], []
for split_name in ["train_2020", "validation_2020"]:
    for ex in ds[split_name]:
        cardiff_texts.append(ex["text"])
        cardiff_labels.append(int(ex["label"][NEWS_IDX]))

print(f"cardiffnlp pool: {len(cardiff_texts)} examples, "
      f"{sum(cardiff_labels)} positive (news_&_social_concern), "
      f"{len(cardiff_labels) - sum(cardiff_labels)} negative")

## Combine, vectorize, train

`TfidfVectorizer`'s `stop_words=None` and `token_pattern=r"\b\w+\b"` are deliberate deviations from sklearn's defaults, matching `category_model.py`'s config — required for train/inference parity through the ONNX export, not style. Reverting either reintroduces a token/bigram mismatch between sklearn's own tokenization and skl2onnx's exported graph.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_texts = cardiff_texts + [r["text"] for r in train_233]
train_labels = cardiff_labels + [int(r["is_political"]) for r in train_233]
print(f"Combined training set: {len(train_texts)} examples "
      f"({sum(train_labels)} political, {len(train_labels) - sum(train_labels)} non-political)")

test_texts = [r["text"] for r in test_233]
test_labels = [int(r["is_political"]) for r in test_233]

train_norm = [text_normalize.normalize_text(t) for t in train_texts]
test_norm = [text_normalize.normalize_text(t) for t in test_texts]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), stop_words=None, sublinear_tf=True, min_df=2,
    token_pattern=r"\b\w+\b",
)
X_train = vectorizer.fit_transform(train_norm)
X_test = vectorizer.transform(test_norm)
print("vocab size:", len(vectorizer.vocabulary_))

clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(X_train, train_labels)
print("trained.")

## Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_probs = clf.predict_proba(X_test)[:, 1]  # P(political)
preds_05 = (test_probs >= 0.5).astype(int)
print(classification_report(test_labels, preds_05, target_names=["non-political", "political"], zero_division=0))
print("confusion matrix (rows=gold, cols=predicted), labels=[non-political, political]:")
print(confusion_matrix(test_labels, preds_05))

## Threshold sweep

Threshold is hand-picked from the sweep below, not a formula — the label mix and number of competing classes shape the probability distribution, so a value from a different run or label set doesn't transfer. Recorded in `config.json` for reference; this notebook doesn't wire it into any exclude or devalue decision.

In [ ]:
sweep_results = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds = (test_probs >= t).astype(int)
    tp = int(((preds == 1) & (np.array(test_labels) == 1)).sum())
    fp = int(((preds == 1) & (np.array(test_labels) == 0)).sum())
    fn = int(((preds == 0) & (np.array(test_labels) == 1)).sum())
    tn = int(((preds == 0) & (np.array(test_labels) == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    sweep_results.append({"threshold": t, "precision": precision, "recall": recall, "f1": f1,
                           "tp": tp, "fp": fp, "fn": fn, "tn": tn})
    print(f"  t={t:.2f}  precision={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}  "
          f"(tp={tp} fp={fp} fn={fn} tn={tn})")

CONFIDENCE_THRESHOLD = 0.5  # confirm against the sweep printed above before publishing

## Spot-check

In [ ]:
spot_checks = [
    "The senate voted 51-49 to pass the appropriations bill today",
    "Just adopted the sweetest rescue puppy, she's already best friends with the cat!",
    "Thanks Donny 👑✝️🤡 #maga #grift",
    "Tried a new ramen spot downtown and it was incredible",
    "My daughter registered to vote for the first time today, so proud",
    "New indie album dropped and it's already on repeat",
    "The White House loves 60 Minutes, per the latest reporting",
    "Woke up early, went for a long walk, feeling great today",
]
for text in spot_checks:
    vec = vectorizer.transform([text_normalize.normalize_text(text)])
    prob = clf.predict_proba(vec)[0][1]
    pred = "political" if prob >= CONFIDENCE_THRESHOLD else "non-political"
    print(f"[{pred}] ({prob:.2f}) {text}")

## Export to ONNX

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType
from sklearn.pipeline import Pipeline
import onnxruntime as ort

pipeline = Pipeline([("tfidf", vectorizer), ("clf", clf)])
onnx_model = convert_sklearn(
    pipeline,
    initial_types=[("input", StringTensorType([None, 1]))],
    options={id(clf): {"zipmap": False}},
)
onnx_bytes = onnx_model.SerializeToString()

sess = ort.InferenceSession(onnx_bytes, providers=["CPUExecutionProvider"])
sample = np.array([[t] for t in spot_checks], dtype=object)
onnx_out = sess.run(None, {"input": sample})
output_names = [o.name for o in sess.get_outputs()]
assert output_names[1] == "probabilities", f"unexpected ONNX output order: {output_names}"
onnx_probs = onnx_out[1]
sklearn_probs = clf.predict_proba(vectorizer.transform([text_normalize.normalize_text(t) for t in spot_checks]))
# atol=0.03: same tolerance category_classifier.ipynb uses, comfortably above the
# sublinear_tf gap's known noise floor but still tight enough to catch a genuinely
# missing term/bigram.
assert np.allclose(onnx_probs, sklearn_probs, atol=0.03), "ONNX/sklearn parity check failed"
assert onnx_probs.shape[1] == 2, "ONNX output width doesn't match the 2 labels"
print("ONNX export verified: parity OK, output width matches 2 labels")

with open("model.onnx", "wb") as f:
    f.write(onnx_bytes)

## Package `config.json`

In [ ]:
import json
from datetime import datetime, timezone

VERSION = "v1"

config = {
    "version": VERSION,
    "text_normalize_source_commit": TEXT_NORMALIZE_COMMIT,
    "labels": ["non-political", "political"],
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "tfidf": {"ngram_range": [1, 2], "stop_words": None, "sublinear_tf": True, "min_df": 2},
    "dataset_composition": {
        "sources": ["cardiffnlp/tweet_topic_multi (train_2020+validation_2020, news_&_social_concern label)",
                     "training/political_tone_eval.jsonl (#233, 60% train split)"],
        "cardiffnlp_pool_size": len(cardiff_texts),
        "cardiffnlp_positive": int(sum(cardiff_labels)),
        "combined_train_examples": len(train_texts),
        "held_out_check_examples": len(test_233),
        "held_out_check_political": int(sum(test_labels)),
        "held_out_check_nonpolitical": len(test_labels) - int(sum(test_labels)),
    },
    "threshold_sweep": sweep_results,
    "mistake_budget_eval_note": (
        "See #235 for the full mistake-budget comparison against #234's baselines on "
        "this same held-out check set."
    ),
    "trained_at": datetime.now(timezone.utc).isoformat(),
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))

## Upload to R2

Uploads always happen unconditionally — versioned artifacts are cheap and safe to publish. Promoting a version to production (flipping `latest.json`) is a separate, deliberate step, not done here — nothing in `processing/` loads this prefix yet.

In [ ]:
R2_MODELS_ACCOUNT_ID = R2_MODELS_ACCESS_KEY_ID = R2_MODELS_SECRET_ACCESS_KEY = R2_MODELS_BUCKET_NAME = None

try:
    from google.colab import userdata

    R2_MODELS_ACCOUNT_ID = userdata.get("R2_MODELS_ACCOUNT_ID")
    R2_MODELS_ACCESS_KEY_ID = userdata.get("R2_MODELS_ACCESS_KEY_ID")
    R2_MODELS_SECRET_ACCESS_KEY = userdata.get("R2_MODELS_SECRET_ACCESS_KEY")
    R2_MODELS_BUCKET_NAME = userdata.get("R2_MODELS_BUCKET_NAME")
except Exception as e:
    print(f"Colab secrets unavailable ({type(e).__name__}: {e}), trying Kaggle secrets...")

if not R2_MODELS_ACCOUNT_ID:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        R2_MODELS_ACCOUNT_ID = secrets.get_secret("R2_MODELS_ACCOUNT_ID")
        R2_MODELS_ACCESS_KEY_ID = secrets.get_secret("R2_MODELS_ACCESS_KEY_ID")
        R2_MODELS_SECRET_ACCESS_KEY = secrets.get_secret("R2_MODELS_SECRET_ACCESS_KEY")
        R2_MODELS_BUCKET_NAME = secrets.get_secret("R2_MODELS_BUCKET_NAME")
    except Exception as e:
        print(f"Kaggle secrets unavailable ({type(e).__name__}: {e}), falling back to manual values...")

if not R2_MODELS_ACCOUNT_ID:
    # Manual fallback -- fill these in locally, never commit real values.
    R2_MODELS_ACCOUNT_ID = ""
    R2_MODELS_ACCESS_KEY_ID = ""
    R2_MODELS_SECRET_ACCESS_KEY = ""
    R2_MODELS_BUCKET_NAME = ""

assert (
    R2_MODELS_ACCOUNT_ID
    and R2_MODELS_ACCESS_KEY_ID
    and R2_MODELS_SECRET_ACCESS_KEY
    and R2_MODELS_BUCKET_NAME
), "R2 credentials not set -- see the markdown cell above"

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_MODELS_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_MODELS_ACCESS_KEY_ID,
    aws_secret_access_key=R2_MODELS_SECRET_ACCESS_KEY,
    region_name="auto",
)

PREFIX = f"political-classifier/{VERSION}"
s3.upload_file("model.onnx", R2_MODELS_BUCKET_NAME, f"{PREFIX}/model.onnx")
s3.upload_file("config.json", R2_MODELS_BUCKET_NAME, f"{PREFIX}/config.json")
print(f"uploaded to s3://{R2_MODELS_BUCKET_NAME}/{PREFIX}/")

## Promote to production

Not promoted here. Promotion is a deliberate, separate step, run once something actually loads this model:
```
cd training && uv run python r2_release.py --model political publish v1
```